# LeetCode 742 — Closest Leaf in a Binary Tree bằng Python

Link đề: https://leetcode.com/problems/closest-leaf-in-a-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def findClosestLeaf(self, root: Optional[TreeNode], k: int) -> int:
        ...
```

Ghi chú: đây là bài **Premium** (cần tài khoản trả phí để xem trên LeetCode). Đề trong notebook này được viết theo hiểu biết về bài (có đối chiếu bản mô tả tiếng Anh), nên nếu bạn có tài khoản Premium hãy **đối chiếu lại đề gốc**. Các ví dụ gốc có hình vẽ cây; notebook vẽ lại bằng chữ — hãy đối chiếu hình gốc nếu có điều kiện.

Điểm mấu chốt: "gần nhất" tính theo **số cạnh đi trên cây**, và đường đi có thể **đi lên cha rồi mới đi xuống**. Vì vậy cây phải được coi như một **đồ thị vô hướng**. Notebook cũng tự dựng lớp `TreeNode` và hàm dựng cây từ danh sách theo thứ tự tầng kiểu LeetCode (`[1,2,3,None,4]`), vì LeetCode cho cây dưới dạng đó.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho cây nhị phân có giá trị các nút **đôi một khác nhau** và một giá trị `k` nằm trong cây; tìm **giá trị của lá gần nhất** với nút `k` (gần = ít cạnh nhất, được phép đi lên rồi đi xuống).
2. Nắm công cụ cần dùng: lớp `TreeNode`, hàm dựng cây từ danh sách tầng có `None`, định nghĩa **lá**, bản đồ `cha`, `deque` cho BFS, tập `visited`, và cách coi cây như đồ thị vô hướng.
3. Đi từ **tính khoảng cách qua đường gốc `O(n·h)`** → **BFS trên đồ thị cây có con trỏ cha** → **quy hoạch động "lá gần nhất bên dưới" cộng khoảng cách tới tổ tiên** → **BFS đa nguồn xuất phát từ mọi lá**.
4. Hiểu mấu chốt: lá gần nhất có thể nằm ở nhánh khác của một **tổ tiên**; chỉ tìm trong cây con của `k` là sai.
5. Tránh các lỗi kinh điển: chỉ tìm xuống dưới, nhận diện lá sai (`or` thay vì `and`), bỏ qua chính `k`, BFS không có `visited`, và dựng cây sai khi đọc danh sách kiểu LeetCode.

## 2. Đề bài & ví dụ

Cho `root` của một cây nhị phân mà **mọi nút có giá trị khác nhau**, và một số nguyên `k`. Trong cây tồn tại một nút có `val == k`. Hãy trả về **giá trị của lá gần nhất** với nút đó.

"Gần nhất" nghĩa là **số cạnh đi qua ít nhất** trên cây để từ nút `k` tới một lá. **Lá** là nút **không có con nào**.

**Ví dụ 1**

- Input: `root = [1,3,2]`, `k = 1`
- Output: `2` (hoặc `3` — cả hai lá đều cách gốc 1 cạnh, đề chấp nhận một trong hai).

**Ví dụ 2**

- Input: `root = [1]`, `k = 1`
- Output: `1` (gốc chính là lá).

**Ví dụ 3**

- Input: `root = [1,2,3,4,null,null,null,5,null,6]`, `k = 2`
- Output: `3`
- Giải thích: lá `3` cách nút `2` hai cạnh (`2 → 1 → 3`), còn lá `6` cách ba cạnh (`2 → 4 → 5 → 6`). Chú ý đường tới lá `3` phải **đi lên** nút cha `1`.

**Ràng buộc:**

- Số nút trong khoảng `[1, 1000]`.
- `1 <= Node.val <= 1000`, mọi giá trị khác nhau.
- Tồn tại nút có `Node.val == k`.

Vì có thể có **nhiều đáp án đúng** (hòa nhau, như ví dụ 1), khi kiểm thử ta dùng hàm kiểm tra "đáp án có thuộc tập các lá gần nhất hay không" thay vì so sánh cứng.

In [1]:
from typing import List, Optional
from collections import deque, defaultdict
import random
import sys

sys.setrecursionlimit(20000)       # cây lệch 1000 nút cần đệ quy sâu; xem mục 3.5
inf = float("inf")

## 3. Kiến thức nền

### 3.1. Lớp `TreeNode` và danh sách tầng kiểu LeetCode

LeetCode mô tả cây bằng danh sách **duyệt theo tầng** (level order), dùng `null` (Python: `None`) cho con vắng. Cách đọc: lấy nút gốc; rồi lần lượt với mỗi nút trong hàng đợi lấy **hai** phần tử kế tiếp làm con trái, con phải (nếu khác `None` thì tạo nút và đưa vào hàng đợi). Phần tử `None` **không** được xếp vào hàng đợi (nó không có con).

Ví dụ `[1,2,3,4,None,None,None,5,None,6]`: nút `1` nhận `2,3`; nút `2` nhận `4,None`; nút `3` nhận `None,None`; nút `4` nhận `5,None`; nút `5` nhận `6,None`.

In [2]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


def build_tree(values):
    """Dựng cây từ danh sách tầng kiểu LeetCode (None = vắng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            q.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> danh sách tầng, bỏ các None thừa ở cuối."""
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        q.append(node.left)
        q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


ex3 = [1, 2, 3, 4, None, None, None, 5, None, 6]
root3 = build_tree(ex3)
print("gốc:", root3, "| con trái:", root3.left, "| con phải:", root3.right)
print("2.left =", root3.left.left, "| 2.right =", root3.left.right)
print("khứ hồi:", tree_to_list(root3) == ex3)

gốc: TreeNode(1) | con trái: TreeNode(2) | con phải: TreeNode(3)
2.left = TreeNode(4) | 2.right = None
khứ hồi: True


Để nhìn cây cho trực quan, ta in cây theo từng tầng (`.` là chỗ vắng, theo vị trí trong cây đầy đủ):

In [3]:
def show_tree(root):
    """In cây theo tầng; mỗi phần tử là giá trị hoặc '.' cho vị trí vắng (chỉ in tới tầng sâu nhất)."""
    level = [root]
    d = 1
    while any(x is not None for x in level):
        print(f"tầng {d}:", " ".join(str(x.val) if x else "." for x in level))
        nxt = []
        for x in level:
            nxt.append(x.left if x else None)
            nxt.append(x.right if x else None)
        level = nxt
        d += 1


show_tree(root3)

tầng 1: 1
tầng 2: 2 3
tầng 3: 4 . . .
tầng 4: 5 . . . . . . .
tầng 5: 6 . . . . . . . . . . . . . . .


### 3.2. Định nghĩa **lá** và bẫy `or`

Nút là **lá** khi **không có con trái VÀ không có con phải**: `node.left is None and node.right is None`. Nút chỉ thiếu **một** con **không** phải lá. (Mẹo ngắn của doocs: với hai con đều `None`, `node.left == node.right` là `True`; nhưng hai con khác nhau thì `False`, nên cách này đúng chỉ vì hai nút khác nhau không bằng nhau — ta dùng cách viết rõ ràng cho dễ hiểu.)

In [4]:
def is_leaf(node):
    return node.left is None and node.right is None


def leaves_of(root):
    """Danh sách các lá (theo thứ tự trước-trái-phải), duyệt lặp bằng ngăn xếp."""
    out, stack = [], [root]
    while stack:
        node = stack.pop()
        if is_leaf(node):
            out.append(node)
        else:
            if node.right:
                stack.append(node.right)
            if node.left:
                stack.append(node.left)
    return out


print("lá của ví dụ 3:", [x.val for x in leaves_of(root3)])
print("2 có phải lá không?", is_leaf(root3.left), "| 3 có phải lá không?", is_leaf(root3.right))

lá của ví dụ 3: [6, 3]
2 có phải lá không? False | 3 có phải lá không? True


### 3.3. Bản đồ cha (`parent`) — để "đi lên"

Nút `TreeNode` chỉ trỏ **xuống** con, không trỏ lên cha. Để đi lên ta lập một dict `parent[node]`: duyệt cây một lượt, ghi cha cho từng con. Khóa dict là **chính đối tượng nút** (hash theo danh tính đối tượng), nên hoạt động tốt kể cả khi giá trị trùng.

In [5]:
def build_parent(root):
    parent = {root: None}
    stack = [root]
    while stack:
        node = stack.pop()
        for child in (node.left, node.right):
            if child is not None:
                parent[child] = node
                stack.append(child)
    return parent


parent = build_parent(root3)
print({n.val: (p.val if p else None) for n, p in parent.items()})

{1: None, 2: 1, 3: 1, 4: 2, 5: 4, 6: 5}


### 3.4. BFS với `deque` và `visited`

BFS (duyệt theo chiều rộng) mở rộng từ nguồn theo **từng lớp khoảng cách**: nút ở lớp `d` được lấy ra trước nút ở lớp `d + 1`. Vì vậy **lần đầu** ta gặp một nút thỏa điều kiện (là lá) thì nó có khoảng cách ngắn nhất. `deque` cho `popleft()` trong `O(1)`; `visited` (một `set`) tránh đi vòng, vì đồ thị cây vô hướng có cạnh hai chiều giữa cha và con.

In [6]:
q = deque([1])
q.append(2)
q.append(3)
print("popleft:", q.popleft(), q.popleft(), "| còn:", list(q))
seen = {1}
for x in (1, 2, 1, 3, 2):
    print(x, "đã thấy" if x in seen else "mới")
    seen.add(x)

popleft: 1 2 | còn: [3]
1 đã thấy
2 mới
1 đã thấy
3 mới
2 đã thấy


### 3.5. Độ sâu đệ quy và `sys.setrecursionlimit`

Cây `1000` nút có thể là một **dây chuyền** sâu `1000`. Đệ quy của Python mặc định giới hạn khoảng `1000` tầng nên có thể gây `RecursionError`. Hai cách phòng: (1) viết **lặp** bằng ngăn xếp (như `build_parent`, cách nộp của ta); (2) tăng giới hạn bằng `sys.setrecursionlimit(...)` (đã làm ở cell đầu của notebook, cho các cách minh họa đệ quy).

In [7]:
def make_chain(n, side="left"):
    """Dây chuyền n nút: 1 -> 2 -> ... -> n theo phía 'left' hoặc 'right'."""
    nodes = [TreeNode(i) for i in range(1, n + 1)]
    for a, b in zip(nodes, nodes[1:]):
        if side == "left":
            a.left = b
        else:
            a.right = b
    return nodes[0]


chain = make_chain(1000)
print("độ sâu dây chuyền:", len(leaves_of(chain)), "lá; lá duy nhất có giá trị", leaves_of(chain)[0].val)

độ sâu dây chuyền: 1 lá; lá duy nhất có giá trị 1000


## 4. Ý tưởng nền tảng / trực giác

Vẽ cây ví dụ 3 và nút `k = 2`:

```
tầng 1:  1
tầng 2:  2   3
tầng 3:  4   .   .   .
tầng 4:  5
tầng 5:  6
```

Các lá: `3` và `6`. Khoảng cách từ `2`:

- tới `6`: đi **xuống** `2 → 4 → 5 → 6`: 3 cạnh.
- tới `3`: đi **lên** `2 → 1` rồi **xuống** `1 → 3`: 2 cạnh.

Nếu chỉ nhìn **cây con của `2`** ta chỉ thấy lá `6` và trả lời sai. Do đó phải cho phép **đi lên**. Một cây nhị phân cùng các cạnh cha-con (hai chiều) là một **đồ thị vô hướng không có chu trình**; bài toán trở thành: "từ nút `k` trong đồ thị đó, tìm **lá gần nhất**" — đúng bài BFS đường ngắn nhất trên đồ thị không trọng số.

Công thức khoảng cách giữa hai nút `a, b`: `dist(a, b) = depth(a) + depth(b) - 2 * depth(LCA(a, b))`, hay đơn giản hơn: đi theo đường gốc → `a` và gốc → `b`, bỏ đoạn chung.

In [8]:
def path_to(root, target_val):
    """Đường từ gốc tới nút có giá trị target_val (list các nút), duyệt lặp."""
    stack = [(root, [root])]
    while stack:
        node, path = stack.pop()
        if node.val == target_val:
            return path
        for child in (node.left, node.right):
            if child is not None:
                stack.append((child, path + [child]))
    return None


def tree_dist(root, a, b):
    pa, pb = path_to(root, a), path_to(root, b)
    common = 0
    while common < min(len(pa), len(pb)) and pa[common] is pb[common]:
        common += 1
    return (len(pa) - common) + (len(pb) - common)


print("dist(2, 6) =", tree_dist(root3, 2, 6), "| dist(2, 3) =", tree_dist(root3, 2, 3), "| dist(2, 2) =", tree_dist(root3, 2, 2))

dist(2, 6) = 3 | dist(2, 3) = 2 | dist(2, 2) = 0


## 5. Cách 1 — Brute Force: tính khoảng cách tới từng lá qua đường gốc

### Ý tưởng

1. Tìm **mọi lá**.
2. Với mỗi lá, tính khoảng cách tới `k` bằng công thức đường gốc ở mục 4 (dùng `path_to`, bỏ đoạn chung).
3. Chọn lá có khoảng cách nhỏ nhất.

Cách này đơn giản, đúng chắc chắn, và ta cũng dùng nó làm **oracle** (trả về cả **tập mọi lá gần nhất** để kiểm tra các đáp án có nhiều lựa chọn).

In [9]:
def nearest_leaves(root, k):
    """Oracle: tập các giá trị lá có khoảng cách nhỏ nhất tới k (kèm khoảng cách)."""
    best, who = inf, set()
    for leaf in leaves_of(root):
        d = tree_dist(root, k, leaf.val)
        if d < best:
            best, who = d, {leaf.val}
        elif d == best:
            who.add(leaf.val)
    return best, who


def closest_leaf_paths(root, k):
    return min(nearest_leaves(root, k)[1])        # chọn giá trị nhỏ nhất trong các lá hòa


cases = [
    ([1, 3, 2], 1, {2, 3}),
    ([1], 1, {1}),
    (ex3, 2, {3}),
]
for lst, k, allowed in cases:
    r = build_tree(lst)
    d, who = nearest_leaves(r, k)
    print(lst, "k =", k, "| khoảng cách", d, "| các lá gần nhất:", who, "| chọn:", closest_leaf_paths(r, k))
    assert who == allowed

[1, 3, 2] k = 1 | khoảng cách 1 | các lá gần nhất: {2, 3} | chọn: 2
[1] k = 1 | khoảng cách 0 | các lá gần nhất: {1} | chọn: 1
[1, 2, 3, 4, None, None, None, 5, None, 6] k = 2 | khoảng cách 2 | các lá gần nhất: {3} | chọn: 3


### Độ phức tạp và nút thắt

- Số lá `L <= n`; mỗi lần tính khoảng cách tìm hai đường dài tối đa `h` (chiều cao) kèm sao chép đường: `O(n·h)` cho `path_to` mỗi lá, tức `O(L · n)` thực tế trong cài đặt ngây thơ (mỗi `path_to` có thể duyệt cả cây). Tổng `O(n^2)` trong trường hợp xấu; bộ nhớ `O(n)`.
- Với `n = 1000` vẫn chạy được, nhưng ta làm đi làm lại cùng một việc: mỗi lá lại **tìm lại đường từ gốc**.

Câu hỏi: có cách nào chỉ **lan từ `k` ra xung quanh** một lần và dừng ngay khi gặp lá đầu tiên không? Đó chính là BFS.

## 6. Ý tưởng thuật toán / bước đột phá

Coi cây là **đồ thị vô hướng** (cạnh cha ↔ con) rồi chạy **BFS từ nút `k`**:

1. Lập `parent` để biết cách đi lên.
2. Hàng đợi bắt đầu chỉ có nút `k`, `visited = {k}`.
3. Mỗi lần lấy một nút ra: nếu nó là **lá** → trả giá trị của nó (BFS đảm bảo đây là lá gần nhất).
4. Ngược lại đẩy các hàng xóm chưa thăm: `left`, `right`, `parent`.

Mỗi nút vào hàng đợi tối đa một lần nên tổng `O(n)`. Hai chi tiết quan trọng: `k` **tự nó có thể là lá** (kiểm tra ngay khi lấy ra, không bỏ qua nguồn), và phải có `visited` để không quay lại cha.

## 7. Các cách tiếp theo

Tổng cộng 4 cách: (1) khoảng cách qua đường gốc (mục 5), (2) BFS trên đồ thị cây có con trỏ cha, (3) DP "lá gần nhất bên dưới" + tổ tiên, (4) BFS đa nguồn từ mọi lá.

### 7.1. Cách 2 — Con trỏ cha + BFS từ `k` (`O(n)`)

**Giải thích từng đoạn.**

1. `parent = build_parent(root)` và tìm nút mục tiêu `target` (nút có `val == k`).
2. `q = deque([target])`, `seen = {target}`.
3. Lặp: `node = q.popleft()`; nếu là lá thì `return node.val`.
4. Với `nxt` trong `(node.left, node.right, parent[node])`: nếu `nxt` không phải `None` và chưa thăm thì đánh dấu và đẩy vào `q`.

In [10]:
def closest_leaf_bfs(root, k):
    parent = build_parent(root)
    target = next(n for n in parent if n.val == k)
    q = deque([target])
    seen = {target}
    while q:
        node = q.popleft()
        if node.left is None and node.right is None:
            return node.val
        for nxt in (node.left, node.right, parent[node]):
            if nxt is not None and nxt not in seen:
                seen.add(nxt)
                q.append(nxt)


def check_answer(root, k, got):
    return got in nearest_leaves(root, k)[1]


for lst, k, allowed in cases:
    r = build_tree(lst)
    got = closest_leaf_bfs(r, k)
    print(lst, "k =", k, "->", got)
    assert got in allowed

[1, 3, 2] k = 1 -> 3
[1] k = 1 -> 1
[1, 2, 3, 4, None, None, None, 5, None, 6] k = 2 -> 3


**Code nộp LeetCode (cách 2, lặp — không lo giới hạn đệ quy):**

```python
from collections import deque
from typing import Optional

class Solution:
    def findClosestLeaf(self, root: Optional[TreeNode], k: int) -> int:
        parent = {root: None}
        target = None
        stack = [root]
        while stack:
            node = stack.pop()
            if node.val == k:
                target = node
            for child in (node.left, node.right):
                if child is not None:
                    parent[child] = node
                    stack.append(child)
        q = deque([target])
        seen = {target}
        while q:
            node = q.popleft()
            if node.left is None and node.right is None:
                return node.val
            for nxt in (node.left, node.right, parent[node]):
                if nxt is not None and nxt not in seen:
                    seen.add(nxt)
                    q.append(nxt)
```

**Độ phức tạp:** Time `O(n)`, Space `O(n)` (dict cha, `seen`, hàng đợi).

### 7.2. Cách 3 — DP "lá gần nhất bên dưới" + duyệt các tổ tiên của `k`

**Ý tưởng khác hẳn (không dùng BFS).**

Gọi `down[x] = (độ sâu tới lá gần nhất trong cây con của x, giá trị lá đó)`, tính **từ dưới lên**: lá có `(0, val)`; nút khác lấy `min` hai con rồi cộng 1 vào khoảng cách.

Đường từ `k` tới một lá đi **lên** tổ tiên `a` nào đó (có thể chính `k`, khi `a = k` thì đi thẳng xuống) rồi **xuống** theo một nhánh. Do đó với mỗi tổ tiên `a` của `k` (kể cả `k`), ứng viên là `dist(k, a) + down[a].dist`. Lấy `min` qua mọi tổ tiên.

Vì sao không sai khi `down[a]` có thể đi xuống đúng nhánh chứa `k`? Khi đó đường ứng viên là một **đường đi (walk) dài hơn đường thật** tới cùng lá ấy, nên ứng viên đó chỉ **lớn hơn hoặc bằng** khoảng cách thật tới một lá; còn khoảng cách tối ưu thật sự (qua tổ tiên chung thấp nhất của `k` và lá tối ưu) cũng được xét. Do đó `min` vẫn bằng khoảng cách nhỏ nhất.

In [11]:
def closest_leaf_dp(root, k):
    down = {}

    def go(node):
        if node is None:
            return (inf, None)
        if node.left is None and node.right is None:
            down[node] = (0, node.val)
            return down[node]
        a, b = go(node.left), go(node.right)
        best = min(a, b)                       # so (khoảng cách, giá trị) -> ưu tiên khoảng cách nhỏ
        down[node] = (best[0] + 1, best[1])
        return down[node]

    go(root)
    path = path_to(root, k)                    # gốc -> k
    best = (inf, None)
    for steps_up, anc in enumerate(reversed(path)):      # steps_up = dist(k, anc)
        d, leaf = down[anc]
        best = min(best, (steps_up + d, leaf))
    return best[1]


for lst, k, allowed in cases:
    r = build_tree(lst)
    got = closest_leaf_dp(r, k)
    print(lst, "k =", k, "->", got)
    assert got in allowed

[1, 3, 2] k = 1 -> 2
[1] k = 1 -> 1
[1, 2, 3, 4, None, None, None, 5, None, 6] k = 2 -> 3


**Code nộp LeetCode (cách 3):**

```python
class Solution:
    def findClosestLeaf(self, root: Optional[TreeNode], k: int) -> int:
        inf = float("inf")
        down = {}

        def go(node):
            if node is None:
                return (inf, None)
            if node.left is None and node.right is None:
                down[node] = (0, node.val)
                return down[node]
            best = min(go(node.left), go(node.right))
            down[node] = (best[0] + 1, best[1])
            return down[node]

        go(root)

        # tìm đường gốc -> k (lặp)
        stack = [(root, [root])]
        path = None
        while stack:
            node, p = stack.pop()
            if node.val == k:
                path = p
                break
            for c in (node.left, node.right):
                if c is not None:
                    stack.append((c, p + [c]))

        best = (inf, None)
        for steps_up, anc in enumerate(reversed(path)):
            d, leaf = down[anc]
            best = min(best, (steps_up + d, leaf))
        return best[1]
```

**Độ phức tạp:** Time `O(n)` cho DP và `O(n·h)` nếu sao chép đường (có thể giảm xuống `O(n)` bằng con trỏ cha), Space `O(n)`. Lưu ý: hàm `go` đệ quy, `h` tới 1000 có thể cần `setrecursionlimit` trên máy bạn.

### 7.3. Cách 4 — BFS đa nguồn từ **mọi lá** vào trong

**Ý tưởng khác hẳn.** Đảo ngược câu hỏi: thay vì từ `k` đi tìm lá, ta cho **mọi lá cùng xuất phát** (khoảng cách 0), mỗi lá mang **nhãn** là chính giá trị của nó. BFS lan ra: nút nào được chạm tới **đầu tiên** sẽ nhận nhãn của lá đã chạm tới nó — đó là lá gần nhất của nút ấy. Đến khi lan tới `k`, ta trả nhãn của `k`.

Ưu điểm: **một lần BFS cho đáp án của MỌI nút** (xem biến thể 15.2).

In [12]:
def build_adj(root):
    """Đồ thị vô hướng của cây: dict nút -> list hàng xóm."""
    adj = defaultdict(list)
    stack = [root]
    while stack:
        node = stack.pop()
        adj[node]                                  # đảm bảo có khóa dù là cây 1 nút
        for child in (node.left, node.right):
            if child is not None:
                adj[node].append(child)
                adj[child].append(node)
                stack.append(child)
    return adj


def closest_leaf_multi(root, k):
    adj = build_adj(root)
    label = {}                                      # nút -> giá trị lá gần nhất
    q = deque()
    for node in adj:
        if node.left is None and node.right is None:
            label[node] = node.val
            q.append(node)
    while q:
        node = q.popleft()
        if node.val == k:
            return label[node]
        for nxt in adj[node]:
            if nxt not in label:
                label[nxt] = label[node]
                q.append(nxt)


for lst, k, allowed in cases:
    r = build_tree(lst)
    got = closest_leaf_multi(r, k)
    print(lst, "k =", k, "->", got)
    assert got in allowed

[1, 3, 2] k = 1 -> 3
[1] k = 1 -> 1
[1, 2, 3, 4, None, None, None, 5, None, 6] k = 2 -> 3


**Code nộp LeetCode (cách 4):**

```python
from collections import deque, defaultdict
from typing import Optional

class Solution:
    def findClosestLeaf(self, root: Optional[TreeNode], k: int) -> int:
        adj = defaultdict(list)
        stack = [root]
        while stack:
            node = stack.pop()
            adj[node]
            for child in (node.left, node.right):
                if child is not None:
                    adj[node].append(child)
                    adj[child].append(node)
                    stack.append(child)
        label = {}
        q = deque()
        for node in adj:
            if node.left is None and node.right is None:
                label[node] = node.val
                q.append(node)
        while q:
            node = q.popleft()
            if node.val == k:
                return label[node]
            for nxt in adj[node]:
                if nxt not in label:
                    label[nxt] = label[node]
                    q.append(nxt)
```

**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

Bốn cách cho cùng đáp án (hoặc đồng nghĩa khi hòa); mục 12 sẽ đối chiếu bằng hàm kiểm tra "thuộc tập lá gần nhất".

## 8. TRACE từng bước

Trace cách 2 (BFS) trên ví dụ 3, `k = 2`. Thứ tự hàng xóm là `left, right, parent`.

| bước | lấy ra | là lá? | hàng đợi sau khi đẩy hàng xóm mới |
|---|---|---|---|
| 0 | - | - | `[2]` |
| 1 | `2` | không | `[4, 1]` (trái `4`, phải vắng, cha `1`) |
| 2 | `4` | không | `[1, 5]` (trái `5`; cha `2` đã thăm) |
| 3 | `1` | không | `[5, 3]` (trái `2` đã thăm; phải `3`) |
| 4 | `5` | không | `[3, 6]` (trái `6`) |
| 5 | `3` | **lá** | trả `3` |

Lá `3` được lấy ra **trước** lá `6` vì cách `k` 2 cạnh so với 3 cạnh — đúng tính chất BFS (xếp theo khoảng cách).

In [13]:
def trace_bfs(root, k):
    parent = build_parent(root)
    target = next(n for n in parent if n.val == k)
    q = deque([(target, 0)])
    seen = {target}
    print(f"BFS từ {k}:")
    step = 0
    while q:
        node, d = q.popleft()
        step += 1
        leaf = node.left is None and node.right is None
        print(f"bước {step}: lấy {node.val} (khoảng cách {d}) | lá? {leaf}", end="")
        if leaf:
            print(f"  -> TRẢ {node.val}")
            return node.val
        added = []
        for nxt in (node.left, node.right, parent[node]):
            if nxt is not None and nxt not in seen:
                seen.add(nxt)
                q.append((nxt, d + 1))
                added.append(nxt.val)
        print(f" | đẩy thêm {added} | hàng đợi {[(n.val, dd) for n, dd in q]}")


assert trace_bfs(build_tree(ex3), 2) == 3

BFS từ 2:
bước 1: lấy 2 (khoảng cách 0) | lá? False | đẩy thêm [4, 1] | hàng đợi [(4, 1), (1, 1)]
bước 2: lấy 4 (khoảng cách 1) | lá? False | đẩy thêm [5] | hàng đợi [(1, 1), (5, 2)]
bước 3: lấy 1 (khoảng cách 1) | lá? False | đẩy thêm [3] | hàng đợi [(5, 2), (3, 2)]
bước 4: lấy 5 (khoảng cách 2) | lá? False | đẩy thêm [6] | hàng đợi [(3, 2), (6, 3)]
bước 5: lấy 3 (khoảng cách 2) | lá? True  -> TRẢ 3


Trace thêm ví dụ 1 (`[1,3,2]`, `k = 1`; hòa giữa `3` và `2`) và ví dụ 2 (`[1]`; gốc là lá).

In [14]:
r = trace_bfs(build_tree([1, 3, 2]), 1)
print("-> chọn", r, "(đề chấp nhận 2 hoặc 3)")
print()
assert trace_bfs(build_tree([1]), 1) == 1

BFS từ 1:
bước 1: lấy 1 (khoảng cách 0) | lá? False | đẩy thêm [3, 2] | hàng đợi [(3, 1), (2, 1)]
bước 2: lấy 3 (khoảng cách 1) | lá? True  -> TRẢ 3
-> chọn 3 (đề chấp nhận 2 hoặc 3)

BFS từ 1:
bước 1: lấy 1 (khoảng cách 0) | lá? True  -> TRẢ 1


## 9. Code nộp LeetCode

Đây là cách 2 (con trỏ cha + BFS, hoàn toàn lặp). Ta dùng `TreeNode` tự dựng ở mục 3.1 để chạy thử như LeetCode.

```python
from collections import deque
from typing import Optional

class Solution:
    def findClosestLeaf(self, root: Optional[TreeNode], k: int) -> int:
        parent = {root: None}
        target = None
        stack = [root]
        while stack:
            node = stack.pop()
            if node.val == k:
                target = node
            for child in (node.left, node.right):
                if child is not None:
                    parent[child] = node
                    stack.append(child)
        q = deque([target])
        seen = {target}
        while q:
            node = q.popleft()
            if node.left is None and node.right is None:
                return node.val
            for nxt in (node.left, node.right, parent[node]):
                if nxt is not None and nxt not in seen:
                    seen.add(nxt)
                    q.append(nxt)
```

In [15]:
class Solution:
    def findClosestLeaf(self, root: Optional[TreeNode], k: int) -> int:
        parent = {root: None}
        target = None
        stack = [root]
        while stack:
            node = stack.pop()
            if node.val == k:
                target = node
            for child in (node.left, node.right):
                if child is not None:
                    parent[child] = node
                    stack.append(child)
        q = deque([target])
        seen = {target}
        while q:
            node = q.popleft()
            if node.left is None and node.right is None:
                return node.val
            for nxt in (node.left, node.right, parent[node]):
                if nxt is not None and nxt not in seen:
                    seen.add(nxt)
                    q.append(nxt)


sol = Solution()
for lst, k, allowed in cases:
    got = sol.findClosestLeaf(build_tree(lst), k)
    print(lst, "k =", k, "->", got, "| chấp nhận:", sorted(allowed))
    assert got in allowed

[1, 3, 2] k = 1 -> 3 | chấp nhận: [2, 3]
[1] k = 1 -> 1 | chấp nhận: [1]
[1, 2, 3, 4, None, None, None, 5, None, 6] k = 2 -> 3 | chấp nhận: [3]


## 10. Vì sao đúng

**Phát biểu.** BFS từ nút `k` trên đồ thị vô hướng của cây trả về một lá có khoảng cách nhỏ nhất tới `k`.

**Chứng minh ngắn.**

1. Đồ thị gồm các nút của cây, cạnh nối mỗi nút với cha của nó. Khoảng cách giữa hai nút trên cây (số cạnh của đường đi duy nhất giữa chúng) bằng khoảng cách ngắn nhất trong đồ thị này (cây không có chu trình nên mọi đường đơn là duy nhất).
2. BFS từ `k` thăm các nút **theo thứ tự không giảm của khoảng cách**: trước tiên khoảng cách 0, rồi 1, rồi 2... Chứng minh quy nạp: nút ở khoảng cách `d + 1` chỉ được đưa vào hàng đợi khi một nút ở khoảng cách `d` được lấy ra, mà mọi nút khoảng cách `d` được lấy ra trước các nút khoảng cách `d + 1`.
3. Vì vậy **lá đầu tiên** bị lấy ra khỏi hàng đợi có khoảng cách nhỏ nhất tới `k` trong số mọi lá.
4. `visited` đảm bảo mỗi nút vào hàng đợi một lần, nên thuật toán kết thúc và tốn `O(n)`.
5. Có ít nhất một lá (cây hữu hạn, khác rỗng) và đồ thị liên thông nên BFS luôn gặp một lá.

**Với cách 3:** `min` trên mọi tổ tiên `a` của `k` của `dist(k, a) + down[a]` không bao giờ nhỏ hơn khoảng cách thật tới lá tương ứng (đó là độ dài một đường đi tồn tại), và bằng khoảng cách thật khi `a` là tổ tiên chung thấp nhất của `k` và lá gần nhất.

**Với cách 4:** BFS đa nguồn gán cho mỗi nút nhãn của lá chạm tới nó đầu tiên, nên nhãn là lá gần nhất của nút đó.

## 11. Time/Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Khoảng cách qua đường gốc cho từng lá | `O(n^2)` (xấu nhất) | `O(n)` | oracle |
| 2. Con trỏ cha + BFS từ `k` (`Solution`) | `O(n)` | `O(n)` | cách nộp |
| 3. DP lá gần nhất bên dưới + tổ tiên | `O(n)` (+ `O(n·h)` nếu sao chép đường) | `O(n)` | không dùng BFS |
| 4. BFS đa nguồn từ mọi lá | `O(n)` | `O(n)` | cho đáp án của mọi nút |

`n <= 1000` nên mọi cách chạy tức thì; lưu ý đệ quy sâu của các cách có `go(...)` đệ quy trên cây dây chuyền `1000` nút (đã xử lý bằng `setrecursionlimit` và cách nộp viết lặp).

## 12. Bộ test

Chiến lược: (a) ví dụ của đề và kiểm tra **hàm dựng cây**, (b) edge case (1 nút, `k` là lá, `k` là gốc, dây chuyền trái/phải, cây đầy đủ), (c) phản ví dụ cho các lời giải sai, (d) đối chiếu **mọi cách** với **hai oracle độc lập** (khoảng cách qua đường gốc và Floyd–Warshall) trên cây ngẫu nhiên có seed, (e) cây lớn `n = 1000`.

Vì có thể có nhiều lá hòa nhau, ta kiểm tra bằng hàm `check_answer(root, k, got)`: đáp án **phải là một trong các lá gần nhất**.

### 12.1. Ví dụ của đề và kiểm tra hàm dựng cây

In [16]:
ALL = {
    "paths": closest_leaf_paths,
    "bfs": closest_leaf_bfs,
    "dp": closest_leaf_dp,
    "multi": closest_leaf_multi,
    "solution": lambda root, k: Solution().findClosestLeaf(root, k),
}

# 1) hàm dựng cây: khứ hồi danh sách <-> cây
roundtrips = [
    [1], [1, 2], [1, None, 3], [1, 3, 2], [1, 2, 3, 4, None, None, None, 5, None, 6],
    [5, 3, 8, 1, 4, 7, 9], [1, 2, None, 3, None, 4], [1, None, 2, None, 3],
]
for lst in roundtrips:
    assert tree_to_list(build_tree(lst)) == lst, lst
assert build_tree([]) is None and build_tree([None]) is None
print("hàm dựng cây khứ hồi đúng với", len(roundtrips), "danh sách")

# 2) ví dụ của đề
for lst, k, allowed in cases:
    r = build_tree(lst)
    for name, fn in ALL.items():
        got = fn(r, k)
        assert got in allowed, (name, lst, k, got)
        assert check_answer(r, k, got)
print("ví dụ của đề đúng với", len(ALL), "cách")

hàm dựng cây khứ hồi đúng với 8 danh sách
ví dụ của đề đúng với 5 cách


### 12.2. Edge case

In [17]:
edge = [
    ([1], 1),                                        # một nút: chính nó
    ([1, 2], 1),                                     # gốc chỉ có con trái: lá duy nhất là 2
    ([1, None, 3], 1),                               # gốc chỉ có con phải
    ([1, 2, 3], 2),                                  # k là lá
    ([1, 2, 3], 1),                                  # hòa nhau
    ([5, 3, 8, 1, 4, 7, 9], 5),                      # cây đầy đủ 3 tầng, k = gốc
    ([5, 3, 8, 1, 4, 7, 9], 3),                      # k ở giữa
    ([1, 2, None, 3, None, 4], 4),                   # dây chuyền trái, k là lá cuối
    ([1, 2, None, 3, None, 4], 1),                   # dây chuyền trái, k là gốc
    ([1, None, 2, None, 3, None, 4], 2),             # dây chuyền phải
    (ex3, 1), (ex3, 4), (ex3, 5), (ex3, 6), (ex3, 3),
]
for lst, k in edge:
    r = build_tree(lst)
    for name, fn in ALL.items():
        got = fn(r, k)
        assert check_answer(r, k, got), (name, lst, k, got)
print("edge case: tất cả", len(edge), "ca đúng với", len(ALL), "cách")

# đáp án cụ thể cho vài ca đã biết
assert closest_leaf_bfs(build_tree([1, 2]), 1) == 2
assert closest_leaf_bfs(build_tree([1, 2, None, 3, None, 4]), 1) == 4
assert closest_leaf_bfs(build_tree(ex3), 6) == 6

edge case: tất cả 15 ca đúng với 5 cách


### 12.3. Hai oracle độc lập và cây ngẫu nhiên nhỏ

Oracle thứ hai dùng **Floyd–Warshall** trên đồ thị cây (không dùng `path_to`). Hai oracle phải cho cùng tập lá gần nhất; rồi ta đối chiếu mọi cách với chúng. Cây ngẫu nhiên có giá trị **khác nhau** (đúng ràng buộc) và cả cây lệch sâu.

In [18]:
def random_tree(rng, n, deep=False):
    vals = rng.sample(range(1, 1001), n)
    root = TreeNode(vals[0])
    slots = [(root, "left"), (root, "right")]
    for v in vals[1:]:
        if deep:
            i = rng.randrange(max(0, len(slots) - 3), len(slots))     # thiên về nút mới -> cây sâu
        else:
            i = rng.randrange(len(slots))
        node, side = slots[i]
        slots[i] = slots[-1]
        slots.pop()
        new = TreeNode(v)
        setattr(node, side, new)
        slots.append((new, "left"))
        slots.append((new, "right"))
    return root


def all_nodes(root):
    out, stack = [], [root]
    while stack:
        x = stack.pop()
        out.append(x)
        for c in (x.left, x.right):
            if c is not None:
                stack.append(c)
    return out


def floyd_nearest(root, k):
    nodes = all_nodes(root)
    idx = {n: i for i, n in enumerate(nodes)}
    m = len(nodes)
    d = [[inf] * m for _ in range(m)]
    for i in range(m):
        d[i][i] = 0
    for n in nodes:
        for c in (n.left, n.right):
            if c is not None:
                d[idx[n]][idx[c]] = d[idx[c]][idx[n]] = 1
    for t in range(m):
        for i in range(m):
            for j in range(m):
                if d[i][t] + d[t][j] < d[i][j]:
                    d[i][j] = d[i][t] + d[t][j]
    s = next(i for i, n in enumerate(nodes) if n.val == k)
    leaves = [i for i, n in enumerate(nodes) if n.left is None and n.right is None]
    best = min(d[s][i] for i in leaves)
    return best, {nodes[i].val for i in leaves if d[s][i] == best}


rng = random.Random(742)
count = 0
for _ in range(400):
    n = rng.randint(1, 12)
    root = random_tree(rng, n, deep=rng.random() < 0.4)
    for node in all_nodes(root):
        k = node.val
        o1 = nearest_leaves(root, k)
        o2 = floyd_nearest(root, k)
        assert o1 == o2, (tree_to_list(root), k, o1, o2)
        for name, fn in ALL.items():
            assert fn(root, k) in o1[1], (name, tree_to_list(root), k)
        count += 1
print("2 oracle trùng nhau và mọi cách đúng trên", count, "cặp (cây, k)")

2 oracle trùng nhau và mọi cách đúng trên 2565 cặp (cây, k)


### 12.4. Cây ngẫu nhiên cỡ vừa (n tới 60) với mọi `k`

In [19]:
rng = random.Random(1742)
count = 0
for _ in range(120):
    n = rng.randint(10, 60)
    root = random_tree(rng, n, deep=rng.random() < 0.5)
    nodes = all_nodes(root)
    for node in rng.sample(nodes, min(8, len(nodes))):
        k = node.val
        who = nearest_leaves(root, k)[1]
        for name, fn in ALL.items():
            assert fn(root, k) in who, (name, tree_to_list(root), k)
        count += 1
print("đã kiểm", count, "cặp (cây, k) cỡ vừa")

đã kiểm 960 cặp (cây, k) cỡ vừa


### 12.5. Cây lớn `n = 1000` (đúng ràng buộc)

Gồm cây ngẫu nhiên, cây đầy đủ, dây chuyền trái/phải và cây zig-zag (dây chuyền đổi phía). Các hàm lặp xử lý ổn; các hàm đệ quy dùng `setrecursionlimit` đã đặt.

In [20]:
import time


def make_zigzag(n):
    nodes = [TreeNode(i) for i in range(1, n + 1)]
    for i in range(n - 1):
        if i % 2 == 0:
            nodes[i].left = nodes[i + 1]
        else:
            nodes[i].right = nodes[i + 1]
    return nodes[0]


def make_complete(n):
    nodes = [TreeNode(i) for i in range(1, n + 1)]
    for i in range(n):
        if 2 * i + 1 < n:
            nodes[i].left = nodes[2 * i + 1]
        if 2 * i + 2 < n:
            nodes[i].right = nodes[2 * i + 2]
    return nodes[0]


t0 = time.perf_counter()
rng = random.Random(1000)
big_trees = {
    "ngẫu nhiên": random_tree(rng, 1000),
    "ngẫu nhiên sâu": random_tree(rng, 1000, deep=True),
    "đầy đủ": make_complete(1000),
    "dây chuyền trái": make_chain(1000, "left"),
    "dây chuyền phải": make_chain(1000, "right"),
    "zig-zag": make_zigzag(1000),
}
for name, root in big_trees.items():
    nodes = all_nodes(root)
    assert len(nodes) == 1000
    for node in rng.sample(nodes, 5):
        k = node.val
        who = nearest_leaves(root, k)[1]
        for fname, fn in ALL.items():
            assert fn(root, k) in who, (name, fname, k)
    print(f"{name:16s} n = 1000 đúng với", len(ALL), "cách")
print("thời gian:", round(time.perf_counter() - t0, 2), "s")

# dây chuyền: k ở đầu -> lá duy nhất là nút cuối
assert Solution().findClosestLeaf(make_chain(1000), 1) == 1000
assert Solution().findClosestLeaf(make_chain(1000, "right"), 500) == 1000

ngẫu nhiên       n = 1000 đúng với 5 cách


ngẫu nhiên sâu   n = 1000 đúng với 5 cách


đầy đủ           n = 1000 đúng với 5 cách
dây chuyền trái  n = 1000 đúng với 5 cách
dây chuyền phải  n = 1000 đúng với 5 cách
zig-zag          n = 1000 đúng với 5 cách
thời gian: 8.13 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ tìm lá trong **cây con** của `k`

In [21]:
def wrong_down_only(root, k):
    target = next(n for n in all_nodes(root) if n.val == k)
    q = deque([target])
    while q:
        node = q.popleft()
        if node.left is None and node.right is None:
            return node.val
        for c in (node.left, node.right):          # SAI: không bao giờ đi lên cha
            if c is not None:
                q.append(c)


r3 = build_tree(ex3)
print("sai :", wrong_down_only(r3, 2), "| đúng:", closest_leaf_bfs(r3, 2))
assert wrong_down_only(r3, 2) == 6 and closest_leaf_bfs(r3, 2) == 3

sai : 6 | đúng: 3


**Vì sao sai:** ở ví dụ 3, lá `6` nằm trong cây con của `2` nhưng cách 3 cạnh; lá `3` cách 2 cạnh nhưng phải **đi lên** cha `1` rồi xuống. **Cách đúng:** coi cây là đồ thị vô hướng (có con trỏ cha).

### Lỗi 2 — Nhận diện lá sai: dùng `or` thay vì `and`

In [22]:
def wrong_leaf_or(root, k):
    parent = build_parent(root)
    target = next(n for n in parent if n.val == k)
    q = deque([target])
    seen = {target}
    while q:
        node = q.popleft()
        if node.left is None or node.right is None:      # SAI: chỉ thiếu MỘT con đã coi là lá
            return node.val
        for nxt in (node.left, node.right, parent[node]):
            if nxt is not None and nxt not in seen:
                seen.add(nxt)
                q.append(nxt)


print("sai :", wrong_leaf_or(r3, 2), "| đúng:", closest_leaf_bfs(r3, 2))
assert wrong_leaf_or(r3, 2) == 2 and closest_leaf_bfs(r3, 2) == 3

sai : 2 | đúng: 3


**Vì sao sai:** nút `2` có con trái `4` nhưng thiếu con phải; với `or` nó bị coi là lá (khoảng cách 0). Lá phải **không có con nào**. **Cách đúng:** `node.left is None and node.right is None`.

### Lỗi 3 — Bỏ qua chính `k` (nguồn BFS có thể là lá)

In [23]:
def wrong_skip_self(root, k):
    parent = build_parent(root)
    target = next(n for n in parent if n.val == k)
    seen = {target}
    q = deque()
    for nxt in (target.left, target.right, parent[target]):   # SAI: không kiểm tra chính target
        if nxt is not None:
            seen.add(nxt)
            q.append(nxt)
    while q:
        node = q.popleft()
        if node.left is None and node.right is None:
            return node.val
        for nxt in (node.left, node.right, parent[node]):
            if nxt is not None and nxt not in seen:
                seen.add(nxt)
                q.append(nxt)


print("cây [1], k = 1: sai ->", wrong_skip_self(build_tree([1]), 1), "| đúng ->", closest_leaf_bfs(build_tree([1]), 1))
print("cây [1,2,3], k = 2: sai ->", wrong_skip_self(build_tree([1, 2, 3]), 2), "| đúng ->", closest_leaf_bfs(build_tree([1, 2, 3]), 2))
assert wrong_skip_self(build_tree([1]), 1) is None
assert wrong_skip_self(build_tree([1, 2, 3]), 2) == 3 and closest_leaf_bfs(build_tree([1, 2, 3]), 2) == 2

cây [1], k = 1: sai -> None | đúng -> 1
cây [1,2,3], k = 2: sai -> 3 | đúng -> 2


**Vì sao sai:** khi `k` là lá, đáp án là **chính nó** (khoảng cách 0), như ví dụ 2. Nếu bỏ qua nguồn, cây một nút còn trả `None`. **Cách đúng:** đưa `target` vào hàng đợi và kiểm tra lá **khi lấy ra**.

### Lỗi 4 — BFS không có `visited`: đúng nhưng số bước nổ tung

In [24]:
def count_pops(root, k, use_visited):
    parent = build_parent(root)
    target = next(n for n in parent if n.val == k)
    q = deque([target])
    seen = {target}
    pops = 0
    while q:
        node = q.popleft()
        pops += 1
        if node.left is None and node.right is None:
            return pops
        for nxt in (node.left, node.right, parent[node]):
            if nxt is not None and (not use_visited or nxt not in seen):
                seen.add(nxt)
                q.append(nxt)


chain12 = make_chain(14)
print("dây chuyền 14 nút, k = 1: có visited ->", count_pops(chain12, 1, True), "lần lấy ra;",
      "KHÔNG visited ->", count_pops(chain12, 1, False), "lần lấy ra")
assert count_pops(chain12, 1, True) == 14
assert count_pops(chain12, 1, False) > 1000

dây chuyền 14 nút, k = 1: có visited -> 14 lần lấy ra; KHÔNG visited -> 1913 lần lấy ra


**Vì sao sai (về hiệu năng):** đồ thị vô hướng có cạnh **hai chiều**; không đánh dấu thì từ con ta quay lại cha, từ cha lại xuống con... số phần tử hàng đợi tăng theo cấp số nhân theo độ sâu. Trên cây sâu `1000` nút sẽ treo máy. **Cách đúng:** `seen` (tập đã thăm), kiểm tra trước khi đẩy.

### Lỗi 5 — Dựng cây sai khi đọc danh sách kiểu LeetCode (nhầm với chỉ số heap)

In [25]:
def wrong_build_heap(values):
    nodes = [TreeNode(v) if v is not None else None for v in values]
    for i, node in enumerate(nodes):
        if node is None:
            continue
        l, r = 2 * i + 1, 2 * i + 2                 # SAI: danh sách LeetCode KHÔNG chừa chỗ cho con của None
        if l < len(nodes):
            node.left = nodes[l]
        if r < len(nodes):
            node.right = nodes[r]
    return nodes[0]


bad = wrong_build_heap(ex3)
print("dựng sai   :", tree_to_list(bad))
print("dựng đúng  :", tree_to_list(build_tree(ex3)))
assert tree_to_list(bad) != ex3 and tree_to_list(build_tree(ex3)) == ex3

dựng sai   : [1, 2, 3, 4, None, None, None, 5]
dựng đúng  : [1, 2, 3, 4, None, None, None, 5, None, 6]


**Vì sao sai:** trong định dạng LeetCode, các nút `None` **không có con** và **không chiếm chỗ** của con chúng trong danh sách (khác với cây đánh số heap). Cách đọc đúng là dùng hàng đợi như `build_tree`. Dựng sai làm biến mất lá `6` của ví dụ 3.

### Lỗi 6 (bonus) — Giả định giá trị là chỉ số mảng / giả định sai về duy nhất

Đề bảo giá trị **khác nhau**, nhưng nếu tự mình dùng `parent[val]`, đừng giả định `val` nhỏ để dùng mảng (giá trị tới `1000`, ổn với mảng `1001` phần tử nhưng dễ lỡ). Dùng **chính nút** làm khóa dict là an toàn nhất, như `parent[node]` ở trên.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra bài này:**

- "Khoảng cách gần nhất trên cây tới một nút/loại nút đặc biệt" và đường đi có thể **đi lên** → **chuyển cây thành đồ thị vô hướng** rồi BFS.
- Có nhiều "nguồn" (mọi lá) và hỏi cho từng nút → **BFS đa nguồn**.
- Chỉ cần cho một nút → BFS đơn nguồn từ nút đó, dừng khi gặp đích đầu tiên.

**Mẹo nhớ:**

1. Nút `TreeNode` không biết cha → lập `parent` (hoặc `adj`) trước.
2. BFS lấy đích **khi lấy ra khỏi hàng đợi** (hoặc khi đẩy vào, nhất quán), và **bao gồm cả nguồn**.
3. Lá ⇔ `left is None and right is None`.
4. `visited` bắt buộc với đồ thị vô hướng.
5. Danh sách LeetCode đọc bằng **hàng đợi**, không phải chỉ số `2i+1`.
6. Cây sâu 1000 → ưu tiên cài lặp hoặc tăng `setrecursionlimit`.
7. Họ hàng: LeetCode 863 (All Nodes Distance K in Binary Tree), 542 (01 Matrix, BFS đa nguồn), 111 (Minimum Depth), 1161, 310 (Minimum Height Trees).

## 15. Biến thể

### 15.1. Trả về **khoảng cách** tới lá gần nhất (thay vì giá trị)

In [26]:
def closest_leaf_distance(root, k):
    parent = build_parent(root)
    target = next(n for n in parent if n.val == k)
    q = deque([(target, 0)])
    seen = {target}
    while q:
        node, d = q.popleft()
        if node.left is None and node.right is None:
            return d
        for nxt in (node.left, node.right, parent[node]):
            if nxt is not None and nxt not in seen:
                seen.add(nxt)
                q.append((nxt, d + 1))


assert closest_leaf_distance(build_tree(ex3), 2) == 2
assert closest_leaf_distance(build_tree([1]), 1) == 0
rng = random.Random(151)
for _ in range(200):
    root = random_tree(rng, rng.randint(1, 15))
    k = rng.choice(all_nodes(root)).val
    assert closest_leaf_distance(root, k) == nearest_leaves(root, k)[0]
print("15.1 OK")

15.1 OK


### 15.2. Đáp án cho **mọi nút** bằng một lần BFS đa nguồn

Cách 4 tự nhiên mở rộng: lưu cả khoảng cách; sau một lần BFS ta có `(khoảng cách, nhãn lá)` cho **mọi** nút.

In [27]:
def closest_leaf_all(root):
    adj = build_adj(root)
    info = {}                                    # nút -> (khoảng cách, giá trị lá)
    q = deque()
    for node in adj:
        if node.left is None and node.right is None:
            info[node] = (0, node.val)
            q.append(node)
    while q:
        node = q.popleft()
        d, lab = info[node]
        for nxt in adj[node]:
            if nxt not in info:
                info[nxt] = (d + 1, lab)
                q.append(nxt)
    return {n.val: v for n, v in info.items()}


rng = random.Random(152)
for _ in range(200):
    root = random_tree(rng, rng.randint(1, 14), deep=rng.random() < 0.5)
    allinfo = closest_leaf_all(root)
    assert len(allinfo) == len(all_nodes(root))
    for val, (d, lab) in allinfo.items():
        best, who = nearest_leaves(root, val)
        assert d == best and lab in who
print("15.2 OK: một BFS cho mọi nút; ví dụ 3 ->", closest_leaf_all(build_tree(ex3)))

15.2 OK: một BFS cho mọi nút; ví dụ 3 -> {3: (0, 3), 6: (0, 6), 1: (1, 3), 5: (1, 6), 2: (2, 3), 4: (2, 6)}


### 15.3. Mọi nút cách `target` đúng `K` cạnh (LeetCode 863)

Cùng khung "con trỏ cha + BFS", đếm theo lớp.

In [28]:
def nodes_at_distance(root, target_val, K):
    parent = build_parent(root)
    target = next(n for n in parent if n.val == target_val)
    level = [target]
    seen = {target}
    for _ in range(K):
        nxt_level = []
        for node in level:
            for nxt in (node.left, node.right, parent[node]):
                if nxt is not None and nxt not in seen:
                    seen.add(nxt)
                    nxt_level.append(nxt)
        level = nxt_level
    return sorted(n.val for n in level)


def nodes_at_distance_oracle(root, target_val, K):
    return sorted(n.val for n in all_nodes(root) if tree_dist(root, target_val, n.val) == K)


assert nodes_at_distance(build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4]), 5, 2) == [1, 4, 7]
rng = random.Random(153)
for _ in range(200):
    root = random_tree(rng, rng.randint(1, 15), deep=rng.random() < 0.5)
    k = rng.choice(all_nodes(root)).val
    for K in range(0, 6):
        assert nodes_at_distance(root, k, K) == nodes_at_distance_oracle(root, k, K)
print("15.3 OK")

15.3 OK

### 15.4. Độ sâu nhỏ nhất của cây (LeetCode 111)

Độ sâu nhỏ nhất = số nút trên đường ngắn nhất từ gốc tới một lá = (khoảng cách gốc → lá gần nhất) + 1. Ta dùng lại `closest_leaf_distance` tại gốc và đối chiếu với đệ quy.

In [29]:
def min_depth_via_leaf(root):
    return closest_leaf_distance(root, root.val) + 1


def min_depth_rec(node):
    if node is None:
        return 0
    if node.left is None:
        return 1 + min_depth_rec(node.right)
    if node.right is None:
        return 1 + min_depth_rec(node.left)
    return 1 + min(min_depth_rec(node.left), min_depth_rec(node.right))


rng = random.Random(154)
for _ in range(300):
    root = random_tree(rng, rng.randint(1, 20), deep=rng.random() < 0.5)
    assert min_depth_via_leaf(root) == min_depth_rec(root)
print("15.4 OK:", min_depth_via_leaf(build_tree([3, 9, 20, None, None, 15, 7])))

15.4 OK: 2


### 15.5. Giá trị có thể **trùng**: bắt đầu từ chính nút (đối tượng) thay vì giá trị

Đề gốc bảo đảm giá trị duy nhất, nhưng nếu không, ta xuất phát từ **tham chiếu nút** và trả về nút lá.

In [30]:
def closest_leaf_from_node(root, start):
    parent = build_parent(root)
    q = deque([start])
    seen = {start}
    while q:
        node = q.popleft()
        if node.left is None and node.right is None:
            return node
        for nxt in (node.left, node.right, parent[node]):
            if nxt is not None and nxt not in seen:
                seen.add(nxt)
                q.append(nxt)


dup = build_tree([7, 7, 7, 7])                 # giá trị trùng hoàn toàn
leaf = closest_leaf_from_node(dup, dup)
print("lá gần gốc nhất là nút có con trái/phải None:", leaf.left is None and leaf.right is None, "| val =", leaf.val)
assert leaf is dup.right

lá gần gốc nhất là nút có con trái/phải None: True | val = 7


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Khoảng cách qua đường gốc | mỗi lá một lần tính `dist` | `O(n^2)` | `O(n)` | oracle, cây nhỏ |
| 2. Con trỏ cha + BFS từ `k` | cây = đồ thị vô hướng | `O(n)` | `O(n)` | **cách nộp** |
| 3. DP lá bên dưới + tổ tiên | `min` qua tổ tiên của `k` | `O(n)` | `O(n)` | không muốn BFS/`visited` |
| 4. BFS đa nguồn từ lá | lá lan nhãn vào trong | `O(n)` | `O(n)` | cần đáp án cho **nhiều/mọi** `k` |

Khuyến nghị: nộp cách 2; dùng cách 4 khi bài đổi thành nhiều truy vấn; dùng cách 1 + Floyd–Warshall làm oracle khi tự kiểm thử.

In [31]:
rng = random.Random(16)
for _ in range(300):
    root = random_tree(rng, rng.randint(1, 25), deep=rng.random() < 0.5)
    k = rng.choice(all_nodes(root)).val
    who = nearest_leaves(root, k)[1]
    assert who == floyd_nearest(root, k)[1]
    results = {name: fn(root, k) for name, fn in ALL.items()}
    assert all(v in who for v in results.values()), (tree_to_list(root), k, results)
print("300 cây: mọi cách cho đáp án nằm trong tập lá gần nhất")

300 cây: mọi cách cho đáp án nằm trong tập lá gần nhất


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(tree_to_list(build_tree([1, 2, 3, None, 4])))
print([x.val for x in leaves_of(build_tree([1, 2, 3, None, 4]))])
print(closest_leaf_bfs(build_tree([1, 2, 3, None, 4]), 2))
```

In [32]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với cây `[1,2,3,4,None,None,5,6]` (nút `4` có con trái `6`; nút `3` có con phải `5`):

(a) Liệt kê các lá. (b) Đáp án với `k = 1` và khoảng cách. (c) Đáp án với `k = 2` và khoảng cách.

In [33]:
# Bài 2: điền kết quả tính tay
hand_leaves = None           # list giá trị các lá
hand_k1 = None               # (giá trị lá gần nhất, khoảng cách) với k = 1
hand_k2 = None               # (giá trị lá gần nhất, khoảng cách) với k = 2

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành BFS trên cây có con trỏ cha.

In [34]:
def exercise_3_closest(root: Optional[TreeNode], k: int) -> int:
    parent = build_parent(root)
    target = next(n for n in parent if n.val == k)
    q = deque([target])
    seen = {target}
    # TODO 1: lặp khi q còn phần tử
    #     TODO 2: lấy node ra từ đầu hàng đợi (popleft)
    #     TODO 3: nếu node là lá (không có con trái VÀ không có con phải) thì trả node.val
    #     TODO 4: với mỗi hàng xóm trong (node.left, node.right, parent[node]):
    #             nếu khác None và chưa thăm thì đánh dấu thăm và đẩy vào hàng đợi
    return None


# assert exercise_3_closest(build_tree(ex3), 2) == 3
# assert exercise_3_closest(build_tree([1]), 1) == 1
# assert exercise_3_closest(build_tree([1, 2]), 1) == 2

### Bài 4 — Tự viết lại (vừa-khó): khoảng cách tới lá gần nhất cho MỌI nút

Không nhìn lại mục 15.2, hãy tự viết `exercise_4_all(root)` trả về dict `{giá trị nút: khoảng cách tới lá gần nhất}` bằng **BFS đa nguồn**.

```python
exercise_4_all(build_tree([1, 2, 3])) -> {1: 1, 2: 0, 3: 0}
exercise_4_all(build_tree([1])) -> {1: 0}
```

In [35]:
def exercise_4_all(root: Optional[TreeNode]) -> dict:
    # TODO
    pass


# assert exercise_4_all(build_tree([1, 2, 3])) == {1: 1, 2: 0, 3: 0}
# assert exercise_4_all(build_tree([1])) == {1: 0}

### Bài 5 — Mini challenge: các nút cách `target` đúng `K` cạnh (khó)

Viết `exercise_5_dist_k(root, target, K)` trả về **danh sách giá trị đã sắp xếp** các nút cách nút `target` đúng `K` cạnh (đi lên/xuống tự do).

```python
r = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
exercise_5_dist_k(r, 5, 2) -> [1, 4, 7]
exercise_5_dist_k(r, 5, 0) -> [5]
exercise_5_dist_k(r, 5, 9) -> []
```

Gợi ý hướng nghĩ: con trỏ cha + BFS theo lớp, chạy đúng `K` lớp.

In [36]:
def exercise_5_dist_k(root: Optional[TreeNode], target: int, K: int) -> List[int]:
    # TODO
    pass


# r = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
# assert exercise_5_dist_k(r, 5, 2) == [1, 4, 7]
# assert exercise_5_dist_k(r, 5, 0) == [5]
# assert exercise_5_dist_k(r, 5, 9) == []

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `build_tree` đọc theo tầng: gốc `1`; con `2, 3`; rồi `2` nhận `None, 4`.
- Mức 2: `4` là con **phải** của `2`. Lá là các nút không có con: `4` và `3`.
- Mức 3: dòng 1 là `[1, 2, 3, None, 4]`; dòng 2 là `[4, 3]` (duyệt trước, trái trước); dòng 3 là `4` (`k = 2` có con phải `4`, cách 1 cạnh; lá `3` cách 2 cạnh).

**Bài 2**
- Mức 1: vẽ cây: gốc `1`; `2` (trái) và `3` (phải); `4` là con trái của `2`; `5` là con phải của `3`; `6` là con trái của `4`.
- Mức 2: lá là nút không có con.
- Mức 3: (a) `[6, 5]`; (b) `k = 1`: tới `5` là `1 → 3 → 5` = 2 cạnh; tới `6` là `1 → 2 → 4 → 6` = 3 cạnh → `(5, 2)`; (c) `k = 2`: tới `6` là `2 → 4 → 6` = 2 cạnh; tới `5` là `2 → 1 → 3 → 5` = 3 cạnh → `(6, 2)`.

**Bài 3**
- Mức 1: giống mục 7.1: `while q:`.
- Mức 2: `node = q.popleft()`; kiểm tra lá; duyệt ba hàng xóm.
- Mức 3: nhớ `seen.add(nxt)` **trước** khi `q.append(nxt)`, và kiểm tra `nxt is not None`.

**Bài 4**
- Mức 1: đưa **mọi lá** vào hàng đợi với khoảng cách `0`.
- Mức 2: dùng `adj` (cha-con hai chiều); khi lấy `node` ra thì gán khoảng cách `d + 1` cho hàng xóm chưa có khoảng cách.
- Mức 3: trả `{n.val: d for n, d in dist.items()}`.

**Bài 5**
- Mức 1: lập `parent`, tìm nút `target`.
- Mức 2: mỗi lớp là danh sách các nút cách `target` đúng `d` cạnh; lớp tiếp theo gồm hàng xóm chưa thăm.
- Mức 3: lặp `K` lần; trả `sorted(n.val for n in level)`; nếu lớp rỗng sớm thì kết quả `[]`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [37]:
print(tree_to_list(build_tree([1, 2, 3, None, 4])))
print([x.val for x in leaves_of(build_tree([1, 2, 3, None, 4]))])
print(closest_leaf_bfs(build_tree([1, 2, 3, None, 4]), 2))

assert tree_to_list(build_tree([1, 2, 3, None, 4])) == [1, 2, 3, None, 4]
assert [x.val for x in leaves_of(build_tree([1, 2, 3, None, 4]))] == [4, 3]
assert closest_leaf_bfs(build_tree([1, 2, 3, None, 4]), 2) == 4

[1, 2, 3, None, 4]
[4, 3]
4


### Đáp án Bài 2

Cây `[1,2,3,4,None,None,5,6]`:

```
tầng 1: 1
tầng 2: 2 3
tầng 3: 4 . . 5
tầng 4: 6
```

Lá: `6`, `5`. Với `k = 1`: `(5, 2)`. Với `k = 2`: `(6, 2)`.

In [38]:
r2 = build_tree([1, 2, 3, 4, None, None, 5, 6])
show_tree(r2)
assert sorted(x.val for x in leaves_of(r2)) == [5, 6]
assert (closest_leaf_bfs(r2, 1), closest_leaf_distance(r2, 1)) == (5, 2)
assert (closest_leaf_bfs(r2, 2), closest_leaf_distance(r2, 2)) == (6, 2)
print("Bài 2 OK")

tầng 1: 1
tầng 2: 2 3
tầng 3: 4 . . 5
tầng 4: 6 . . . . . . .
Bài 2 OK


### Đáp án Bài 3

In [39]:
def exercise_3_closest_solution(root: Optional[TreeNode], k: int) -> int:
    parent = build_parent(root)
    target = next(n for n in parent if n.val == k)
    q = deque([target])
    seen = {target}
    while q:
        node = q.popleft()
        if node.left is None and node.right is None:
            return node.val
        for nxt in (node.left, node.right, parent[node]):
            if nxt is not None and nxt not in seen:
                seen.add(nxt)
                q.append(nxt)


assert exercise_3_closest_solution(build_tree(ex3), 2) == 3
assert exercise_3_closest_solution(build_tree([1]), 1) == 1
assert exercise_3_closest_solution(build_tree([1, 2]), 1) == 2
rng = random.Random(3)
for _ in range(200):
    root = random_tree(rng, rng.randint(1, 15), deep=rng.random() < 0.5)
    k = rng.choice(all_nodes(root)).val
    assert exercise_3_closest_solution(root, k) in nearest_leaves(root, k)[1]
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [40]:
def exercise_4_all_solution(root: Optional[TreeNode]) -> dict:
    adj = build_adj(root)
    dist = {}
    q = deque()
    for node in adj:
        if node.left is None and node.right is None:
            dist[node] = 0
            q.append(node)
    while q:
        node = q.popleft()
        for nxt in adj[node]:
            if nxt not in dist:
                dist[nxt] = dist[node] + 1
                q.append(nxt)
    return {n.val: d for n, d in dist.items()}


assert exercise_4_all_solution(build_tree([1, 2, 3])) == {1: 1, 2: 0, 3: 0}
assert exercise_4_all_solution(build_tree([1])) == {1: 0}
rng = random.Random(4)
for _ in range(200):
    root = random_tree(rng, rng.randint(1, 14), deep=rng.random() < 0.5)
    got = exercise_4_all_solution(root)
    for val, d in got.items():
        assert d == nearest_leaves(root, val)[0]
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [41]:
def exercise_5_dist_k_solution(root: Optional[TreeNode], target: int, K: int) -> List[int]:
    parent = build_parent(root)
    start = next(n for n in parent if n.val == target)
    level = [start]
    seen = {start}
    for _ in range(K):
        nxt_level = []
        for node in level:
            for nxt in (node.left, node.right, parent[node]):
                if nxt is not None and nxt not in seen:
                    seen.add(nxt)
                    nxt_level.append(nxt)
        level = nxt_level
    return sorted(n.val for n in level)


r5 = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
assert exercise_5_dist_k_solution(r5, 5, 2) == [1, 4, 7]
assert exercise_5_dist_k_solution(r5, 5, 0) == [5]
assert exercise_5_dist_k_solution(r5, 5, 9) == []
rng = random.Random(5)
for _ in range(200):
    root = random_tree(rng, rng.randint(1, 15), deep=rng.random() < 0.5)
    k = rng.choice(all_nodes(root)).val
    for K in range(0, 6):
        assert exercise_5_dist_k_solution(root, k, K) == nodes_at_distance_oracle(root, k, K)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
from collections import deque
from typing import Optional

class Solution:
    def findClosestLeaf(self, root: Optional[TreeNode], k: int) -> int:
        parent = {root: None}                       # 1) con trỏ cha
        target, stack = None, [root]
        while stack:
            node = stack.pop()
            if node.val == k:
                target = node
            for child in (node.left, node.right):
                if child is not None:
                    parent[child] = node
                    stack.append(child)
        q, seen = deque([target]), {target}         # 2) BFS từ k trên đồ thị vô hướng
        while q:
            node = q.popleft()
            if node.left is None and node.right is None:    # lá: KHÔNG có con nào
                return node.val
            for nxt in (node.left, node.right, parent[node]):
                if nxt is not None and nxt not in seen:
                    seen.add(nxt)
                    q.append(nxt)
```

Câu thần chú: **"cây thành đồ thị vô hướng (thêm con trỏ cha); BFS từ `k`; lá đầu tiên lấy ra là đáp án."**

### Checklist trước khi submit

- [ ] Có con trỏ cha (hoặc `adj`) để **đi lên**.
- [ ] Lá = `left is None and right is None` (không dùng `or`).
- [ ] Nguồn `k` được kiểm tra như mọi nút (có thể là lá).
- [ ] Có `seen`, đánh dấu trước khi đẩy vào hàng đợi.
- [ ] Cài lặp để tránh `RecursionError` với cây dây chuyền 1000 nút.
- [ ] Test: một nút, `k` là lá, `k` là gốc, hòa nhau, dây chuyền, ví dụ 3 (đáp án `3`).
- [ ] Hàm dựng cây kiểu LeetCode đọc bằng hàng đợi, không dùng chỉ số heap.
- [ ] Bài Premium: nếu có tài khoản, đối chiếu lại đề gốc và các hình ví dụ.
- [ ] Skeleton đúng: `class Solution`, method `findClosestLeaf(self, root: Optional[TreeNode], k: int) -> int`; độ phức tạp `O(n)`.